# Install Depedensi

In [ ]:
!pip install Sastrawi
!pip install tqdm
! pip install joblib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.7/209.7 kB 4.8 MB/s eta 0:00:00


# Import Library

In [ ]:
import pandas as pd
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
import re
from tqdm.auto import tqdm
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import StackingClassifier
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
import warnings
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, RandomizedSearchCV, GridSearchCV
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix, make_scorer, f1_score
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
import warnings
warnings.filterwarnings('ignore')


# Read Datasets

In [ ]:
data_topik = pd.read_csv('dataset_balanced_topic_6000.csv')
data_topik.head(3)

,full_text,topic,type
0,dicancel driver gosend pas udah nunggu 15 meni...,gosend,saran-kritik
1,anjing tidur di gocar sampe rumah udah terang ???,gocar,pertanyaan
2,terimakasihhh abang gojek kuhhh,gojek,pujian


In [ ]:
data_topik.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6000 entries, 0 to 5999
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   full_text  6000 non-null   object
 1   topic      6000 non-null   object
 2   type       6000 non-null   object
dtypes: object(3)
memory usage: 140.8+ KB


In [ ]:
print(f"jumlah baris dan kolom : {data_topik.shape}")

jumlah baris dan kolom : (6000, 3)


In [ ]:
data_topik.columns = ["full_text" , "topic" , "intention"]

# Exploratory Data Analysis

### missing value

In [ ]:
data_topik.isna().sum()

,0
full_text,0
topic,0
type,0


### distribution class

In [ ]:
count_df = data_topik.value_counts("topic")
count_df_frame = count_df.reset_index()
df = count_df_frame.sort_values('count', ascending=False)
styled_df = df.style.background_gradient(
    subset=['count'],
    cmap='Blues',
    low=0.3,
    high=0.8
).set_properties(
    subset=['topic', 'count'], **{'text-align': 'center', 'font-weight': 'bold'}
)
styled_df

,topic,count
0,gocar,600
1,gofood,600
2,gojek,600
3,gopay,600
4,gosend,600
5,grab,600
6,grabcar,600
7,grabexpress,600
8,grabfood,600
9,ovo,600


In [ ]:
import plotly.express as px

topic_counts = data_topik['topic'].value_counts().reset_index()
topic_counts.columns = ['topic', 'count']

fig = px.bar(
    topic_counts,
    x='topic',
    y='count',
    color='count',
    title='Distribution of Topics'
)

fig.update_layout(
    xaxis_title='Topic',
    yaxis_title='Count',
    title_x=0.5
    font=dict(size=14),
    plot_bgcolor='white',
    bargap=0.2
)
fig.update_xaxes(tickangle=45)
fig.show()


### Wordcloud

In [ ]:
from wordcloud import WordCloud
import plotly.express as px
import matplotlib.pyplot as plt

text = " ".join(data_topik['full_text'])
wc = WordCloud(
    width=1600,
    height=800,
    background_color='white',
    colormap='viridis',
    max_words=300,
    min_font_size=10,
    max_font_size=200
).generate(text)

wc.to_file("wordcloud.png")

fig = px.imshow(plt.imread("wordcloud.png"))
fig.update_layout(
    title="WordCloud of Text Data",
    title_x=0.5,
    xaxis_visible=False,
    yaxis_visible=False,
    dragmode=False,
    plot_bgcolor='white'
)
fig.show()


# Pre - Processing

### normalized

In [ ]:
def clean_text(text):
    if isinstance(text, str):
        text = text.lower()
        emoji_pattern = re.compile(
            "["
            u"\U0001F600-\U0001F64F" # Emotikon
            u"\U0001F300-\U0001F5FF" # Simbol & pictograph
            u"\U0001F680-\U0001F6FF" # Simbol transportasi & peta
            u"\U0001F1E0-\U0001F1FF" # Bendera negara
            u"\U00002700-\U000027BF" # Simbol dingbat
            u"\U000024C2-\U0001F251" # Simbol tambahan
            "]+", flags=re.UNICODE)
        text = emoji_pattern.sub(r'', text) #emoji
        text = re.sub(r'http\S+|www\S+|https\S+', '', text) # URL
        text = re.sub(r'@\w+|#\w+', '', text) # mention dan hashtag
        text = re.sub(r'\d+', '', text) # angka
        text = re.sub(r'[^a-z\s]', '', text) #karakter non-huruf
        text = re.sub(r'\s+', ' ', text).strip() # spasi
    else:
        text = ''
    return text
data_topik['clean_text'] = data_topik['full_text'].apply(clean_text)
data_topik[['full_text', 'clean_text']].head()


,full_text,clean_text
0,dicancel driver gosend pas udah nunggu 15 meni...,dicancel driver gosend pas udah nunggu menit a...
1,anjing tidur di gocar sampe rumah udah terang ???,anjing tidur di gocar sampe rumah udah terang
2,terimakasihhh abang gojek kuhhh,terimakasihhh abang gojek kuhhh
3,@GrabID min mau dm soal keluhan grab express k...,min mau dm soal keluhan grab express kok gak b...
4,GoPay Points kayaknya perlu diperbanyak promo-...,gopay points kayaknya perlu diperbanyak promon...


In [ ]:
import re

ABBREV_MAP = {
    "gk": "tidak", "ga": "tidak", "gak": "tidak", "nggak": "tidak", "ngga": "tidak", "tdk": "tidak",
    "t": "tidak", "tak": "tidak", "enggak": "tidak", "ndak": "tidak", "nda": "tidak",

    "yg": "yang", "yng": "yang", "dgn": "dengan", "dg": "dengan", "dng": "dengan",
    "utk": "untuk", "buat": "untuk", "bwt": "untuk", "buad": "untuk",
    "sm": "sama", "sma": "sama", "ama": "sama", "ma": "sama",
    "aja": "saja", "aj": "saja",
    "jd": "jadi", "jdi": "jadi", "jdih": "jadi",
    "pd": "pada", "pda": "pada",
    "krn": "karena", "karn": "karena", "karna": "karena", "krena": "karena",
    "dr": "dari", "drpd": "daripada", "drpdnya": "daripadanya",
    "trus": "terus", "trs": "terus", "teruss": "terus",
    "tp": "tapi", "tpi": "tapi",
    "bkn": "bukan", "bkan": "bukan",
    "blm": "belum", "blom": "belum",
    "sdh": "sudah", "udh": "sudah", "udah": "sudah", "ud": "sudah", "suda": "sudah", "dh":'sudah',
    "dlm": "dalam", "dalem":"dalam", "dlam":"dalam", "skr": "sekarang", "skg": "sekarang", "skrg":"sekarang",
    "kt": "kita",
    "km": "kamu", "kmu": "kamu", "kam": "kamu", "kamuuh": "kamu",
    "sy": "saya", "aq": "saya", "q": "saya", "gue": "saya", "gw": "saya", "gua": "saya", "gwe": "saya",
    "lg": "lagi", "lgi": "lagi",
    "bgt": "banget", "bgtt": "banget", "bngt": "banget", "bangt": "banget",
    "brp": "berapa", "brapa": "berapa",
    "hrs": "harus", "harusny": "harusnya",
    "kl": "kalau", "klo": "kalau", "klu": "kalau", "kalo": "kalau",
    "kmrn": "kemarin", "kmren": "kemarin",
    "bs": "bisa", "bsa": "bisa", "bsq": "bisa",
    "mo": "mau", "mw": "mau", "mow": "mau", "mwu": "mau",
    "plis": "tolong", "pls": "tolong", "tolonglah": "tolong",
    "trmksih": "terima kasih", "makasih": "terima kasih", "makasii": "terima kasih",
    "makasihh": "terima kasih", "thx": "terima kasih", "thanks": "terima kasih",
    "ok": "oke", "okay": "oke", "okey": "oke", "okayy": "oke", "okee": "oke",
    "bbrp": "beberapa", "bbrapa": "beberapa",
    "org": "orang", "orng": "orang", "org2": "orang-orang", "orng2": "orang-orang",
    "mls": "malas", "males": "malas",
    "ksl": "kesal", "kesel": "kesal",
    "tmn": "teman", "temen": "teman", "tmn2": "teman-teman",
    "smg": "semoga",
    "insyaallah": "insya allah", "inshaallah": "insya allah",
    "astgfirullah": "astaghfirullah", "astaghfirulah": "astaghfirullah",
    "alhamdulilah": "alhamdulillah", "alhamdulila": "alhamdulillah",
    "bismilah": "bismillah", "bismila": "bismillah",
    "amiin": "amin", "aminn": "amin", "aminnn": "amin",
    "gituu": "begitu", "gituuu": "begitu", "gituan": "begituan",
    "nih": "ini", "nie": "ini", "ni": "ini",
    "tu": "itu", "tuh": "itu", "ituu": "itu",
    "ajaib": "ajaib",
    "bnr": "benar", "bener": "benar",
    "bgtu": "begitu",
    "bnyk": "banyak", "byk": "banyak",
    "btw": "ngomong-ngomong",
    "jg": "juga", "jga": "juga",
    "skrng": "sekarang",
    "cm": "cuma", "cma": "cuma", "cuman": "cuma",
    "bsk": "besok", "bsok": "besok",
    "dpt": "dapat", "dapet": "dapat",
    "tmpt": "tempat",
    "trs": "terus",
    "td": "tadi",
    "tpn": "tapi",
    "lbh": "lebih",
    "krg": "kurang",
    "cr":"cari", "cri":"cari",
    "dptin": "dapatkan",
    "bisaaa": "bisa",
    "bnget": "banget",

    "ppk": "pepek", "ngentod": "ngentot", "anj": "anjir",
    "anjir": "anjir", "anjay": "anjir", "bjir": "anjir",
    "jir": "anjir", "bejir": "anjir", "anjg": "anjing",
    "njir": "anjir", "njay": "anjir",
    "asu": "anjing", "asw": "anjing", "anjeng":"anjing",
    "bbi":"babi", "mnyt":"monyet","kntl":"kontol",

    "woww": "wow", "wkwk": "haha", "wkwkwk": "haha", "wkww": "haha",
    "hehe": "haha", "hihi": "haha", "hehehe": "haha", "ahah": "haha",
    "yaa": "ya", "yah": "ya", "yaaa": "ya",
    "lho": "loh", "loh": "loh", "kok": "kenapa",
    "dongg": "dong", "donk": "dong",
    "deh": "deh", "lahh": "lah",
    "siih": "sih", "sihh": "sih",
    "bangettt": "banget", "parahh": "parah",
    "mantul": "mantap betul", "mantapp": "mantap", "mantappu": "mantap",
    "kece": "keren", "ciamik": "bagus",
    "bt": "bad mood", "bete": "bad mood",
    "gabisa": "tidak bisa", "gapapa": "tidak apa-apa", "gpp": "tidak apa-apa",
    "gajadi": "tidak jadi", "gaboleh": "tidak boleh",
    "nggaada": "tidak ada", "gaada": "tidak ada", "gakada": "tidak ada",
    "jln":"jalan", "plsss":"please", "grab express" : "grabexpress", "grab car": "grabcar", "ajg" : "anjing",
    "emg":"memang","emng":"memang",
    "loch":"loh", "suru" : "suruh", "aje": "aja", "banggg":"bang",
    "takutttttt":"takut", "blg":"bilang", "blng":"bilang", "knp":"kenapa"
}

def normalize_abbrev(text):
    if not isinstance(text, str):
        return ""
    words = text.split()
    normalized_words = [ABBREV_MAP.get(w.lower(), w) for w in words]
    return " ".join(normalized_words)

data_topik['normalized_text'] = data_topik['clean_text'].apply(normalize_abbrev)
data_topik[['full_text', 'normalized_text']].head(5)


,full_text,normalized_text
0,dicancel driver gosend pas udah nunggu 15 meni...,dicancel driver gosend pas sudah nunggu menit ...
1,anjing tidur di gocar sampe rumah udah terang ???,anjing tidur di gocar sampe rumah sudah terang
2,terimakasihhh abang gojek kuhhh,terimakasihhh abang gojek kuhhh
3,@GrabID min mau dm soal keluhan grab express k...,min mau dm soal keluhan grab express kenapa ti...
4,GoPay Points kayaknya perlu diperbanyak promo-...,gopay points kayaknya perlu diperbanyak promon...


### Stopword

In [ ]:
data_topik.head(5)

,full_text,topic,type,clean_text,normalized_text
0,dicancel driver gosend pas udah nunggu 15 meni...,gosend,saran-kritik,dicancel driver gosend pas udah nunggu menit a...,dicancel driver gosend pas sudah nunggu menit ...
1,anjing tidur di gocar sampe rumah udah terang ???,gocar,pertanyaan,anjing tidur di gocar sampe rumah udah terang,anjing tidur di gocar sampe rumah sudah terang
2,terimakasihhh abang gojek kuhhh,gojek,pujian,terimakasihhh abang gojek kuhhh,terimakasihhh abang gojek kuhhh
3,@GrabID min mau dm soal keluhan grab express k...,grabexpress,pertanyaan,min mau dm soal keluhan grab express kok gak b...,min mau dm soal keluhan grab express kenapa ti...
4,GoPay Points kayaknya perlu diperbanyak promo-...,gopay,saran-kritik,gopay points kayaknya perlu diperbanyak promon...,gopay points kayaknya perlu diperbanyak promon...


In [ ]:
factory = StopWordRemoverFactory()
stopword_list = factory.get_stop_words()
stopword_set = set(stopword_list)

def remove_stopwords(text):
    if not isinstance(text, str):
        return ""
    words = text.split()
    filtered = [w for w in words if w.lower() not in stopword_set]
    return " ".join(filtered)

data_topik['no_stopword'] = data_topik['normalized_text'].apply(remove_stopwords)
data_topik[['normalized_text', 'no_stopword']].head()

,normalized_text,no_stopword
0,dicancel driver gosend pas sudah nunggu menit ...,dicancel driver gosend pas nunggu menit asli b...
1,anjing tidur di gocar sampe rumah sudah terang,anjing tidur gocar sampe rumah terang
2,terimakasihhh abang gojek kuhhh,terimakasihhh abang gojek kuhhh
3,min mau dm soal keluhan grab express kenapa ti...,min mau dm soal keluhan grab express terima kasih
4,gopay points kayaknya perlu diperbanyak promon...,gopay points kayaknya perlu diperbanyak promon...


### stemming

In [ ]:
data_topik.head(5)

,full_text,topic,type,clean_text,normalized_text,no_stopword
0,dicancel driver gosend pas udah nunggu 15 meni...,gosend,saran-kritik,dicancel driver gosend pas udah nunggu menit a...,dicancel driver gosend pas sudah nunggu menit ...,dicancel driver gosend pas nunggu menit asli b...
1,anjing tidur di gocar sampe rumah udah terang ???,gocar,pertanyaan,anjing tidur di gocar sampe rumah udah terang,anjing tidur di gocar sampe rumah sudah terang,anjing tidur gocar sampe rumah terang
2,terimakasihhh abang gojek kuhhh,gojek,pujian,terimakasihhh abang gojek kuhhh,terimakasihhh abang gojek kuhhh,terimakasihhh abang gojek kuhhh
3,@GrabID min mau dm soal keluhan grab express k...,grabexpress,pertanyaan,min mau dm soal keluhan grab express kok gak b...,min mau dm soal keluhan grab express kenapa ti...,min mau dm soal keluhan grab express terima kasih
4,GoPay Points kayaknya perlu diperbanyak promo-...,gopay,saran-kritik,gopay points kayaknya perlu diperbanyak promon...,gopay points kayaknya perlu diperbanyak promon...,gopay points kayaknya perlu diperbanyak promon...


In [ ]:
tqdm.pandas(desc="Stemming (Sastrawi)")
factory = StemmerFactory()
stemmer = factory.create_stemmer()

def apply_stemming(text):
    if not isinstance(text, str):
        return ""
    return stemmer.stem(text)

data_topik['stemmed_text'] = data_topik['no_stopword'].progress_apply(apply_stemming)


Stemming (Sastrawi):   0%|          | 0/6000 [00:00<?, ?it/s]

In [ ]:
data_topik.to_csv("modelling_data_topic.csv" , index=False)

# Modelling

In [ ]:
data_topik = pd.read_csv("modelling_data_topic.csv")
data_topik.head(3)

,full_text,topic,type,clean_text,normalized_text,no_stopword,stemmed_text
0,dicancel driver gosend pas udah nunggu 15 meni...,gosend,saran-kritik,dicancel driver gosend pas udah nunggu menit a...,dicancel driver gosend pas sudah nunggu menit ...,dicancel driver gosend pas nunggu menit asli b...,dicancel driver gosend pas nunggu menit asli b...
1,anjing tidur di gocar sampe rumah udah terang ???,gocar,pertanyaan,anjing tidur di gocar sampe rumah udah terang,anjing tidur di gocar sampe rumah sudah terang,anjing tidur gocar sampe rumah terang,anjing tidur gocar sampe rumah terang
2,terimakasihhh abang gojek kuhhh,gojek,pujian,terimakasihhh abang gojek kuhhh,terimakasihhh abang gojek kuhhh,terimakasihhh abang gojek kuhhh,terimakasihhh abang gojek kuhhh


### Tf-IDF

In [ ]:
data_topik.dropna(inplace=True)

In [ ]:
tfidf = TfidfVectorizer(
    max_features = 7000,
    min_df=2,
    max_df=0.8,
    ngram_range=(1, 2)
)
X_tfidf = tfidf.fit_transform(data_topik['stemmed_text'])
y = data_topik['topic']

In [ ]:
X_tfidf

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 82898 stored elements and shape (5999, 7000)>

In [ ]:
print(f"Shape TF-IDF: {X_tfidf.shape}")
print(f"Jumlah sampel: {X_tfidf.shape[0]}")
print(f"Jumlah fitur TF-IDF: {X_tfidf.shape[1]}")

Shape TF-IDF: (5999, 7000)
Jumlah sampel: 5999
Jumlah fitur TF-IDF: 7000


## 80 / 20

### Split data

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_tfidf,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [ ]:
print(f"Train set: {X_train.shape[0]} sampel")
print(f"Test set: {X_test.shape[0]} sampel")

Train set: 4799 sampel
Test set: 1200 sampel


In [ ]:
df_X_test = pd.DataFrame.sparse.from_spmatrix(X_test)
df_X_test.head(3)

,0,1,2,3,4,5,6,7,8,9,...,6990,6991,6992,6993,6994,6995,6996,6997,6998,6999
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0.115725,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


### modelling

In [ ]:
lgbm = LGBMClassifier(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=7,
    num_leaves=31,
    random_state=42,
    verbose=-1
)

xgb = XGBClassifier(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=7,
    random_state=42,
    eval_metric='mlogloss',
    verbosity=0
)

base_models = [
    ('lgbm', lgbm),
    ('xgb', xgb),
]

meta_model = LogisticRegression(
    max_iter=1000,
    random_state=42,
    multi_class='multinomial',
    solver='lbfgs'
)

stacking_clf = StackingClassifier(
    estimators=base_models,
    final_estimator=meta_model,
    cv=5,
    n_jobs=-1,
)

In [ ]:
stacking_clf.fit(X_train, y_train)

StackingClassifier(cv=5,
                   estimators=[('lgbm',
                                LGBMClassifier(learning_rate=0.05, max_depth=7,
                                               n_estimators=500,
                                               random_state=42, verbose=-1)),
                               ('xgb',
                                XGBClassifier(base_score=None, booster=None,
                                              callbacks=None,
                                              colsample_bylevel=None,
                                              colsample_bynode=None,
                                              colsample_bytree=None,
                                              device=None,
                                              early_stopping_rounds=None,
                                              enable_categorical=False,
                                              eval_metric='mlogl...
                                              learning_rate=0.05, max_bin=None,
                                              max_cat_threshold=None,
                                              max_cat_to_onehot=None,
                                              max_delta_step=None, max_depth=7,
                                              max_leaves=None,
                                              min_child_weight=None,
                                              missing=nan,
                                              monotone_constraints=None,
                                              multi_strategy=None,
                                              n_estimators=500, n_jobs=None,
                                              num_parallel_tree=None, ...))],
                   final_estimator=LogisticRegression(max_iter=1000,
                                                      multi_class='multinomial',
                                                      random_state=42),
                   n_jobs=-1)

### Evaluation

In [ ]:
y_pred_train = stacking_clf.predict(X_train)
y_pred_test = stacking_clf.predict(X_test)

print("\n--- Train Set Performance ---")
train_accuracy = accuracy_score(y_train, y_pred_train)
print(f"Accuracy: {train_accuracy:.4f}")
print("\n--- Test Set Performance ---")
test_accuracy = accuracy_score(y_test, y_pred_test)
print(f"Accuracy: {test_accuracy:.4f}")
print("\n--- Classification Report (Test Set) ---")
print(classification_report(y_test, y_pred_test))
print("\n--- Confusion Matrix (Test Set) ---")
cm = confusion_matrix(y_test, y_pred_test)
print(cm)


--- Train Set Performance ---
Accuracy: 0.9871

--- Test Set Performance ---
Accuracy: 0.8550

--- Classification Report (Test Set) ---
              precision    recall  f1-score   support

       gocar       0.90      0.87      0.88       120
      gofood       0.92      0.81      0.86       120
       gojek       0.80      0.86      0.83       120
       gopay       0.98      0.93      0.95       120
      gosend       0.97      0.89      0.93       120
        grab       0.64      0.80      0.71       120
     grabcar       0.89      0.82      0.86       120
 grabexpress       0.96      0.92      0.94       120
    grabfood       0.74      0.72      0.73       120
         ovo       0.84      0.93      0.88       120

    accuracy                           0.85      1200
   macro avg       0.86      0.85      0.86      1200
weighted avg       0.86      0.85      0.86      1200


--- Confusion Matrix (Test Set) ---
[[104   0   6   0   0   2   8   0   0   0]
 [  0  97   5   1   1   

In [ ]:
import pickle
with open('stacking_model_8020_topic.pkl', 'wb') as f:
    pickle.dump(stacking_clf, f)
print("Model disimpan: stacking_model_8020_topic.pkl")

with open('tfidf_vectorizer8020_topic.pkl', 'wb') as f:
    pickle.dump(tfidf, f)
print("TF-IDF vectorizer disimpan: tfidf_vectorizer8020_topic.pkl")



Model disimpan: stacking_model_8020_topic.pkl
TF-IDF vectorizer disimpan: tfidf_vectorizer8020_topic.pkl


## 70 / 30

### split data

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_tfidf,
    y,
    test_size=0.3,
    random_state=42,
    stratify=y
)

In [ ]:
print(f"Train set: {X_train.shape[0]} sampel")
print(f"Test set: {X_test.shape[0]} sampel")

Train set: 4199 sampel
Test set: 1800 sampel


In [ ]:
df_X_test = pd.DataFrame.sparse.from_spmatrix(X_test)
df_X_test.head(3)

,0,1,2,3,4,5,6,7,8,9,...,6990,6991,6992,6993,6994,6995,6996,6997,6998,6999
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


### modelling

In [ ]:
lgbm = LGBMClassifier(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=7,
    num_leaves=31,
    random_state=42,
    verbose=-1
)

xgb = XGBClassifier(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=7,
    random_state=42,
    eval_metric='mlogloss',
    verbosity=0
)

base_models = [
    ('lgbm', lgbm),
    ('xgb', xgb),
]

meta_model = LogisticRegression(
    max_iter=1000,
    random_state=42,
    multi_class='multinomial',
    solver='lbfgs'
)

stacking_clf = StackingClassifier(
    estimators=base_models,
    final_estimator=meta_model,
    cv=5,
    n_jobs=-1,
)

In [ ]:
stacking_clf.fit(X_train, y_train)

StackingClassifier(cv=5,
                   estimators=[('lgbm',
                                LGBMClassifier(learning_rate=0.05, max_depth=7,
                                               n_estimators=500,
                                               random_state=42, verbose=-1)),
                               ('xgb',
                                XGBClassifier(base_score=None, booster=None,
                                              callbacks=None,
                                              colsample_bylevel=None,
                                              colsample_bynode=None,
                                              colsample_bytree=None,
                                              device=None,
                                              early_stopping_rounds=None,
                                              enable_categorical=False,
                                              eval_metric='mlogl...
                                              learning_rate=0.05, max_bin=None,
                                              max_cat_threshold=None,
                                              max_cat_to_onehot=None,
                                              max_delta_step=None, max_depth=7,
                                              max_leaves=None,
                                              min_child_weight=None,
                                              missing=nan,
                                              monotone_constraints=None,
                                              multi_strategy=None,
                                              n_estimators=500, n_jobs=None,
                                              num_parallel_tree=None, ...))],
                   final_estimator=LogisticRegression(max_iter=1000,
                                                      multi_class='multinomial',
                                                      random_state=42),
                   n_jobs=-1)

### Evaluation

In [ ]:
y_pred_train = stacking_clf.predict(X_train)
y_pred_test = stacking_clf.predict(X_test)

print("\n--- Train Set Performance ---")
train_accuracy = accuracy_score(y_train, y_pred_train)
print(f"Accuracy: {train_accuracy:.4f}")
print("\n--- Test Set Performance ---")
test_accuracy = accuracy_score(y_test, y_pred_test)
print(f"Accuracy: {test_accuracy:.4f}")
print("\n--- Classification Report (Test Set) ---")
print(classification_report(y_test, y_pred_test))
print("\n--- Confusion Matrix (Test Set) ---")
cm = confusion_matrix(y_test, y_pred_test)
print(cm)


--- Train Set Performance ---
Accuracy: 0.9871

--- Test Set Performance ---
Accuracy: 0.8550

--- Classification Report (Test Set) ---
              precision    recall  f1-score   support

       gocar       0.91      0.89      0.90       180
      gofood       0.92      0.82      0.86       180
       gojek       0.79      0.87      0.83       180
       gopay       0.98      0.92      0.95       180
      gosend       0.96      0.89      0.93       180
        grab       0.62      0.78      0.69       180
     grabcar       0.91      0.82      0.87       180
 grabexpress       0.94      0.91      0.93       180
    grabfood       0.78      0.72      0.75       180
         ovo       0.82      0.93      0.87       180

    accuracy                           0.85      1800
   macro avg       0.86      0.86      0.86      1800
weighted avg       0.86      0.85      0.86      1800


--- Confusion Matrix (Test Set) ---
[[160   0   7   1   0   3   8   0   0   1]
 [  0 147  12   1   1   

In [ ]:
import pickle
with open('stacking_model_7030_topic.pkl', 'wb') as f:
    pickle.dump(stacking_clf, f)
print("Model disimpan: stacking_model_7030_topic.pkl")

with open('tf_idf_vectorizer_7030_topic.pkl', 'wb') as f:
    pickle.dump(tfidf, f)
print("TF-IDF vectorizer disimpan: tf_idf_7030_vectorizer_topic.pkl")



Model disimpan: stacking_model_7030_topic.pkl
TF-IDF vectorizer disimpan: tf_idf_7030_vectorizer_topic.pkl


# Hyper Tuning

In [ ]:
import pandas as pd

data_topik = pd.read_csv("modelling_data_topic.csv")
data_topik.head(3)

,full_text,topic,type,clean_text,normalized_text,no_stopword,stemmed_text
0,dicancel driver gosend pas udah nunggu 15 meni...,gosend,saran-kritik,dicancel driver gosend pas udah nunggu menit a...,dicancel driver gosend pas sudah nunggu menit ...,dicancel driver gosend pas nunggu menit asli b...,dicancel driver gosend pas nunggu menit asli b...
1,anjing tidur di gocar sampe rumah udah terang ???,gocar,pertanyaan,anjing tidur di gocar sampe rumah udah terang,anjing tidur di gocar sampe rumah sudah terang,anjing tidur gocar sampe rumah terang,anjing tidur gocar sampe rumah terang
2,terimakasihhh abang gojek kuhhh,gojek,pujian,terimakasihhh abang gojek kuhhh,terimakasihhh abang gojek kuhhh,terimakasihhh abang gojek kuhhh,terimakasihhh abang gojek kuhhh


### TF - IDF

In [ ]:
data_topik.dropna(inplace=True)

In [ ]:
tfidf = TfidfVectorizer(
    min_df=2,
    max_df=0.8,
    ngram_range=(1, 2)
)

X_tfidf = tfidf.fit_transform(data_topik['stemmed_text'])
y = data_topik['topic']


In [ ]:
X_tfidf

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 93166 stored elements and shape (5999, 12134)>

### split data 70/30

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_tfidf,
    y,
    test_size=0.3,
    random_state=42,
    stratify=y
)

print(f"Train set: {X_train.shape[0]} sampel")
print(f"Test set: {X_test.shape[0]} sampel")


Train set: 4199 sampel
Test set: 1800 sampel


In [ ]:
scorer = make_scorer(f1_score, average='weighted')


#### lightgbm

In [ ]:
lgbm_params = {
    'n_estimators': [100, 200, 300],
    'learning_rate': [0.01, 0.05],
    'max_depth': [5, -1],
    'num_leaves': [31, 50, 70],
}

lgbm_base = LGBMClassifier(random_state=42, verbose=-1)

lgbm_search = GridSearchCV(
    lgbm_base,
    lgbm_params,
    cv=3,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)

lgbm_search.fit(X_train, y_train)
best_lgbm = lgbm_search.best_estimator_


# LGBMClassifier(learning_rate=0.01, max_depth=5, n_estimators=300,
#                random_state=42, verbose=-1)

Fitting 3 folds for each of 36 candidates, totalling 108 fits


In [ ]:
print(best_lgbm)


LGBMClassifier(learning_rate=0.01, max_depth=5, n_estimators=300, 
               random_state=42, verbose=-1)


#### xgboost

In [ ]:
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBClassifier
from sklearn.model_selection import GridSearchCV

label_encoder = LabelEncoder()
y_train_enc = label_encoder.fit_transform(y_train)
y_test_enc = label_encoder.transform(y_test)

xgb_params = {
    'n_estimators': [100, 200, 300],
    'learning_rate': [0.01, 0.5],
    'max_depth': [3, 7, 10],
}

xgb_base = XGBClassifier(
    random_state=42,
    eval_metric='mlogloss',
    verbosity=0
)

xgb_search = GridSearchCV(
    xgb_base,
    xgb_params,
    cv=3,
    scoring="accuracy",
    n_jobs=-1,
    verbose=1
)

xgb_search.fit(X_train, y_train_enc)

best_xgb = xgb_search.best_estimator_

# XGBClassifier(base_score=None, booster=None, callbacks=None,
#               colsample_bylevel=None, colsample_bynode=None,
#               colsample_bytree=None, device=None, early_stopping_rounds=None,
#               enable_categorical=False, eval_metric='mlogloss',
#               feature_types=None, feature_weights=None, gamma=None,
#               grow_policy=None, importance_type=None,
#               interaction_constraints=None, learning_rate=0.5, max_bin=None,
#               max_cat_threshold=None, max_cat_to_onehot=None,
#               max_delta_step=None, max_depth=3, max_leaves=None,
#               min_child_weight=None, missing=nan, monotone_constraints=None,
#               multi_strategy=None, n_estimators=100, n_jobs=None,
#               num_parallel_tree=None, ...)

Fitting 3 folds for each of 18 candidates, totalling 54 fits


In [ ]:
print(best_xgb)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='mlogloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.5, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=3, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=100, n_jobs=None,
              num_parallel_tree=None, ...)


#### logistic regresion

In [ ]:
lr_params = {
    'C': [0.001, 0.01, 0.1, 1, 10, 100],
    'penalty': ['l2'],
    'solver': ['lbfgs', 'saga'],
    'max_iter': [500, 1000, 2000]
}

lr_base = LogisticRegression(
    random_state=42,
    multi_class='multinomial'
)

lr_search = GridSearchCV(
    lr_base,
    lr_params,
    cv=3,
    scoring="accuracy",
    n_jobs=-1,
    verbose=1
)

lr_search.fit(X_train, y_train)
best_lr = lr_search.best_estimator_

# LogisticRegression(C=1, max_iter=500, multi_class='multinomial',
#                    random_state=42)

Fitting 3 folds for each of 36 candidates, totalling 108 fits


In [ ]:
print(best_lr)

LogisticRegression(C=1, max_iter=500, multi_class='multinomial',
                   random_state=42)


In [ ]:
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import StackingClassifier

lgbm = LGBMClassifier(
    learning_rate=0.01,
    max_depth=5,
    n_estimators=300,
    random_state=42,
    verbose=-1
)

xgb = XGBClassifier(
    learning_rate=0.5,
    max_depth=3,
    n_estimators=100,
    eval_metric='mlogloss',
    random_state=42,
    verbosity=0
)

base_models = [
    ('lgbm', lgbm),
    ('xgb', xgb),
]

meta_model = LogisticRegression(
    C=1,
    max_iter=500,
    multi_class='multinomial',
    random_state=42
)

stacking_clf = StackingClassifier(
    estimators=base_models,
    final_estimator=meta_model,
    cv=5,
    n_jobs=-1
)


In [ ]:
stacking_clf.fit(X_train, y_train)

StackingClassifier(cv=5,
                   estimators=[('lgbm',
                                LGBMClassifier(learning_rate=0.01, max_depth=5,
                                               n_estimators=300,
                                               random_state=42, verbose=-1)),
                               ('xgb',
                                XGBClassifier(base_score=None, booster=None,
                                              callbacks=None,
                                              colsample_bylevel=None,
                                              colsample_bynode=None,
                                              colsample_bytree=None,
                                              device=None,
                                              early_stopping_rounds=None,
                                              enable_categorical=False,
                                              eval_metric='mlogl...
                                              learning_rate=0.5, max_bin=None,
                                              max_cat_threshold=None,
                                              max_cat_to_onehot=None,
                                              max_delta_step=None, max_depth=3,
                                              max_leaves=None,
                                              min_child_weight=None,
                                              missing=nan,
                                              monotone_constraints=None,
                                              multi_strategy=None,
                                              n_estimators=100, n_jobs=None,
                                              num_parallel_tree=None, ...))],
                   final_estimator=LogisticRegression(C=1, max_iter=500,
                                                      multi_class='multinomial',
                                                      random_state=42),
                   n_jobs=-1)

#### evaluation hyper tuning 70/30

In [ ]:
y_pred_train = stacking_clf.predict(X_train)
y_pred_test = stacking_clf.predict(X_test)

print("\n--- Train Set Performance ---")
train_accuracy = accuracy_score(y_train, y_pred_train)
print(f"Accuracy: {train_accuracy:.4f}")
print("\n--- Test Set Performance ---")
test_accuracy = accuracy_score(y_test, y_pred_test)
print(f"Accuracy: {test_accuracy:.4f}")
print("\n--- Classification Report (Test Set) ---")
print(classification_report(y_test, y_pred_test))
print("\n--- Confusion Matrix (Test Set) ---")
cm = confusion_matrix(y_test, y_pred_test)
print(cm)


--- Train Set Performance ---
Accuracy: 0.9571

--- Test Set Performance ---
Accuracy: 0.8594

--- Classification Report (Test Set) ---
              precision    recall  f1-score   support

       gocar       0.93      0.89      0.91       180
      gofood       0.95      0.80      0.87       180
       gojek       0.81      0.87      0.84       180
       gopay       0.96      0.92      0.94       180
      gosend       0.95      0.92      0.93       180
        grab       0.63      0.81      0.71       180
     grabcar       0.92      0.82      0.86       180
 grabexpress       0.96      0.90      0.93       180
    grabfood       0.77      0.76      0.76       180
         ovo       0.82      0.92      0.87       180

    accuracy                           0.86      1800
   macro avg       0.87      0.86      0.86      1800
weighted avg       0.87      0.86      0.86      1800


--- Confusion Matrix (Test Set) ---
[[160   0   7   1   1   4   7   0   0   0]
 [  0 144  13   1   1   

In [ ]:
import pickle
with open('stacking_model_7030_hyper_topic.pkl', 'wb') as f:
    pickle.dump(stacking_clf, f)
print("Model disimpan: stacking_model_7030_hyper_topic.pkl")

with open('tfidf_vectorizer7030_hyper_topic.pkl', 'wb') as f:
    pickle.dump(tfidf, f)
print("TF-IDF vectorizer disimpan: tfidf_vectorizer7030_hyper_topic.pkl")

Model disimpan: stacking_model_7030_hyper_topic.pkl
TF-IDF vectorizer disimpan: tfidf_vectorizer7030_hyper_topic.pkl


### split data 80/20

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_tfidf,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"Train set: {X_train.shape[0]} sampel")
print(f"Test set: {X_test.shape[0]} sampel")


Train set: 4799 sampel
Test set: 1200 sampel


In [ ]:
scorer = make_scorer(f1_score, average='weighted')


#### lightgbm

In [ ]:
lgbm_params = {
    'n_estimators': [100, 200, 300],
    'learning_rate': [0.01, 0.05],
    'max_depth': [5, -1],
    'num_leaves': [31, 50, 70],
}

lgbm_base = LGBMClassifier(random_state=42, verbose=-1)

lgbm_search = GridSearchCV(
    lgbm_base,
    lgbm_params,
    cv=3,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)

lgbm_search.fit(X_train, y_train)
best_lgbm = lgbm_search.best_estimator_


# LGBMClassifier(learning_rate=0.01, n_estimators=200, random_state=42,
#                verbose=-1)

Fitting 3 folds for each of 36 candidates, totalling 108 fits


In [ ]:
print(best_lgbm)


LGBMClassifier(learning_rate=0.01, n_estimators=200, random_state=42,
               verbose=-1)


#### xgboost

In [ ]:
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBClassifier
from sklearn.model_selection import GridSearchCV

label_encoder = LabelEncoder()
y_train_enc = label_encoder.fit_transform(y_train)
y_test_enc = label_encoder.transform(y_test)

xgb_params = {
    'n_estimators': [100, 200, 300],
    'learning_rate': [0.01, 0.5],
    'max_depth': [3, 7, 10],
}

xgb_base = XGBClassifier(
    random_state=42,
    eval_metric='mlogloss',
    verbosity=0
)

xgb_search = GridSearchCV(
    xgb_base,
    xgb_params,
    cv=3,
    scoring="accuracy",
    n_jobs=-1,
    verbose=1
)

xgb_search.fit(X_train, y_train_enc)

best_xgb = xgb_search.best_estimator_

# XGBClassifier(base_score=None, booster=None, callbacks=None,
#               colsample_bylevel=None, colsample_bynode=None,
#               colsample_bytree=None, device=None, early_stopping_rounds=None,
#               enable_categorical=False, eval_metric='mlogloss',
#               feature_types=None, feature_weights=None, gamma=None,
#               grow_policy=None, importance_type=None,
#               interaction_constraints=None, learning_rate=0.01, max_bin=None,
#               max_cat_threshold=None, max_cat_to_onehot=None,
#               max_delta_step=None, max_depth=10, max_leaves=None,
#               min_child_weight=None, missing=nan, monotone_constraints=None,
#               multi_strategy=None, n_estimators=300, n_jobs=None,
#               num_parallel_tree=None, ...)

Fitting 3 folds for each of 18 candidates, totalling 54 fits


In [ ]:
print(best_xgb)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='mlogloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.01, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=10, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=300, n_jobs=None,
              num_parallel_tree=None, ...)


#### logistic regresion

In [ ]:
lr_params = {
    'C': [0.001, 0.01, 0.1, 1, 10, 100],
    'penalty': ['l2'],
    'solver': ['lbfgs', 'saga'],
    'max_iter': [500, 1000, 2000]
}

lr_base = LogisticRegression(
    random_state=42,
    multi_class='multinomial'
)

lr_search = GridSearchCV(
    lr_base,
    lr_params,
    cv=3,
    scoring="accuracy",
    n_jobs=-1,
    verbose=1
)

lr_search.fit(X_train, y_train)
best_lr = lr_search.best_estimator_

# LogisticRegression(C=10, max_iter=500, multi_class='multinomial',
#                    random_state=42)

Fitting 3 folds for each of 36 candidates, totalling 108 fits


In [ ]:
print(best_lr)

LogisticRegression(C=10, max_iter=500, multi_class='multinomial',
                   random_state=42)


In [ ]:
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import StackingClassifier

lgbm = LGBMClassifier(
    learning_rate=0.01,
    n_estimators=200,
    random_state=42,
    verbose=-1
)

xgb = XGBClassifier(
    learning_rate=0.01,
    max_depth=10,
    n_estimators=300,
    eval_metric='mlogloss',
    random_state=42,
    verbosity=0
)

base_models = [
    ('lgbm', lgbm),
    ('xgb', xgb),
]

meta_model = LogisticRegression(
    C=10,
    max_iter=500,
    multi_class='multinomial',
    random_state=42
)

stacking_clf = StackingClassifier(
    estimators=base_models,
    final_estimator=meta_model,
    cv=5,
    n_jobs=-1
)


In [ ]:
stacking_clf.fit(X_train, y_train)

StackingClassifier(cv=5,
                   estimators=[('lgbm',
                                LGBMClassifier(learning_rate=0.01,
                                               n_estimators=200,
                                               random_state=42, verbose=-1)),
                               ('xgb',
                                XGBClassifier(base_score=None, booster=None,
                                              callbacks=None,
                                              colsample_bylevel=None,
                                              colsample_bynode=None,
                                              colsample_bytree=None,
                                              device=None,
                                              early_stopping_rounds=None,
                                              enable_categorical=False,
                                              eval_metric='mlogloss',
                                              feature...
                                              learning_rate=0.01, max_bin=None,
                                              max_cat_threshold=None,
                                              max_cat_to_onehot=None,
                                              max_delta_step=None, max_depth=10,
                                              max_leaves=None,
                                              min_child_weight=None,
                                              missing=nan,
                                              monotone_constraints=None,
                                              multi_strategy=None,
                                              n_estimators=300, n_jobs=None,
                                              num_parallel_tree=None, ...))],
                   final_estimator=LogisticRegression(C=10, max_iter=500,
                                                      multi_class='multinomial',
                                                      random_state=42),
                   n_jobs=-1)

#### evaluation hyper tuning 80/20

In [ ]:
y_pred_train = stacking_clf.predict(X_train)
y_pred_test = stacking_clf.predict(X_test)

print("\n--- Train Set Performance ---")
train_accuracy = accuracy_score(y_train, y_pred_train)
print(f"Accuracy: {train_accuracy:.4f}")
print("\n--- Test Set Performance ---")
test_accuracy = accuracy_score(y_test, y_pred_test)
print(f"Accuracy: {test_accuracy:.4f}")
print("\n--- Classification Report (Test Set) ---")
print(classification_report(y_test, y_pred_test))
print("\n--- Confusion Matrix (Test Set) ---")
cm = confusion_matrix(y_test, y_pred_test)
print(cm)


--- Train Set Performance ---
Accuracy: 0.9406

--- Test Set Performance ---
Accuracy: 0.8583

--- Classification Report (Test Set) ---
              precision    recall  f1-score   support

       gocar       0.94      0.87      0.90       120
      gofood       0.93      0.82      0.87       120
       gojek       0.80      0.86      0.83       120
       gopay       0.98      0.93      0.95       120
      gosend       0.96      0.90      0.93       120
        grab       0.64      0.81      0.71       120
     grabcar       0.93      0.82      0.87       120
 grabexpress       0.96      0.90      0.93       120
    grabfood       0.75      0.74      0.74       120
         ovo       0.81      0.94      0.87       120

    accuracy                           0.86      1200
   macro avg       0.87      0.86      0.86      1200
weighted avg       0.87      0.86      0.86      1200


--- Confusion Matrix (Test Set) ---
[[104   0   9   0   0   1   5   0   0   1]
 [  0  98   4   1   1   

In [ ]:
import pickle
with open('stacking_model_8020_hyper_topic.pkl', 'wb') as f:
    pickle.dump(stacking_clf, f)
print("Model disimpan: stacking_model_8020_hyper_topic.pkl")

with open('tfidf_vectorizer8020_hyper_topic.pkl', 'wb') as f:
    pickle.dump(tfidf, f)
print("TF-IDF vectorizer disimpan: tfidf_vectorizer8020_hyper_topic.pkl")

Model disimpan: stacking_model_8020_hyper_topic.pkl
TF-IDF vectorizer disimpan: tfidf_vectorizer8020_hyper_topic.pkl


#